# Report Generation

LLM-based medical report generation using retrieval-augmented generation.

In [ ]:
import subprocess
import json
from openai import OpenAI
from typing import Dict, List

## LLM Client Classes

In [ ]:
class OllamaClient:
    """Local Ollama client for open-source models."""
    
    def __init__(self, model="llama3"):
        self.model = model
    
    def generate(self, prompt, max_tokens=512):
        """Generate text using Ollama."""
        result = subprocess.run(
            ["ollama", "run", self.model],
            input=prompt,
            text=True,
            capture_output=True
        )
        return result.stdout.strip()


class OpenAIClient:
    """OpenAI API client."""
    
    def __init__(self, api_key, model="gpt-4"):
        self.client = OpenAI(api_key=api_key)
        self.model = model
    
    def generate(self, prompt, max_tokens=512):
        """Generate text using OpenAI API."""
        response = self.client.chat.completions.create(
            model=self.model,
            messages=[
                {"role": "system", "content": "You are a radiologist assistant generating chest X-ray reports."},
                {"role": "user", "content": prompt}
            ],
            max_tokens=max_tokens
        )
        return response.choices[0].message.content

## Prompt Templates

In [ ]:
def build_report_prompt(abnormality_score, disease_probs, retrieved_cases, mode="proposed"):
    """
    Build structured prompt for report generation.
    
    Args:
        abnormality_score: Float abnormality score
        disease_probs: Dict of disease probabilities
        retrieved_cases: Dict with 'abnormal' and 'healthy' cases
        mode: 'baseline' (no retrieval) or 'proposed' (with retrieval)
    """
    
    # System instruction
    header = """You are a radiologist assistant. Generate a structured chest X-ray report.
Use ONLY the provided evidence. Do NOT make unsupported claims.
Use cautious clinical language (e.g., "suggests", "consistent with", "may indicate").

"""
    
    # Query case information
    query_block = f"""QUERY CASE:
- Abnormality Score: {abnormality_score:.3f}
- Disease Probabilities:
"""
    
    # Add top disease probabilities
    top_diseases = sorted(disease_probs.items(), key=lambda x: x[1], reverse=True)[:5]
    for disease, prob in top_diseases:
        if prob > 0.1:  # Only show significant probabilities
            query_block += f"  • {disease}: {prob:.3f}\n"
    
    # Evidence from retrieved cases
    evidence_block = "\nRETRIEVED SIMILAR CASES:\n"
    
    if mode == "baseline":
        # Baseline: no retrieval, just use classification
        evidence_block = "\n(No retrieved cases - generate based on disease probabilities only)\n"
    
    else:  # proposed
        # Add abnormal cases
        for i, case in enumerate(retrieved_cases.get('abnormal', [])[:3], 1):
            evidence_block += f"\nAbnormal Case {i} (similarity: {case.get('similarity', 0):.3f}):\n"
            if 'findings' in case:
                evidence_block += f"- Findings: {case['findings'][:200]}\n"
            if 'impression' in case:
                evidence_block += f"- Impression: {case['impression'][:200]}\n"
        
        # Add healthy references
        for i, case in enumerate(retrieved_cases.get('healthy', [])[:2], 1):
            evidence_block += f"\nHealthy Reference {i}:\n"
            if 'impression' in case:
                evidence_block += f"- Impression: {case['impression'][:200]}\n"
    
    # Task instruction
    instruction = """\nTASK:
Generate a structured report with the following sections:

1. FINDINGS: Describe visible abnormalities based on disease probabilities
2. COMPARISON: Compare with retrieved cases (if available)
3. IMPRESSION: Clinical synthesis in 1-2 sentences

Use professional medical language. Avoid overconfident claims.
"""
    
    return header + query_block + evidence_block + instruction

## Report Generator Class

In [ ]:
class ReportGenerator:
    def __init__(self, llm_client):
        """Initialize with an LLM client."""
        self.llm_client = llm_client
    
    def generate(self, abnormality_score, disease_probs, retrieved_cases, mode="proposed"):
        """
        Generate medical report.
        
        Returns:
            str: Generated report
        """
        # Build prompt
        prompt = build_report_prompt(
            abnormality_score,
            disease_probs,
            retrieved_cases,
            mode=mode
        )
        
        # Generate report
        report = self.llm_client.generate(prompt)
        
        return report
    
    def generate_comparison(self, abnormality_score, disease_probs, retrieved_cases):
        """
        Generate both baseline and proposed reports for comparison.
        
        Returns:
            dict: {'baseline': report, 'proposed': report}
        """
        baseline = self.generate(abnormality_score, disease_probs, retrieved_cases, mode="baseline")
        proposed = self.generate(abnormality_score, disease_probs, retrieved_cases, mode="proposed")
        
        return {
            'baseline': baseline,
            'proposed': proposed
        }

## Initialize Report Generator

In [ ]:
# Option 1: Use Ollama (local, free)
llm_client = OllamaClient(model="llama3")  # or "meditron", "biomistral"

# Option 2: Use OpenAI (requires API key)
# llm_client = OpenAIClient(api_key="your-api-key", model="gpt-4")

report_generator = ReportGenerator(llm_client)

## Test Report Generation

In [ ]:
# Example data (replace with actual pipeline output)
test_data = {
    'abnormality_score': 0.68,
    'disease_probs': {
        'Cardiomegaly': 0.72,
        'Pleural Effusion': 0.45,
        'Atelectasis': 0.31,
        'No Finding': 0.15
    },
    'retrieved_cases': {
        'abnormal': [
            {
                'similarity': 0.89,
                'findings': 'Moderate cardiomegaly. Small bilateral pleural effusions.',
                'impression': 'Congestive heart failure.'
            }
        ],
        'healthy': [
            {
                'similarity': 0.45,
                'impression': 'No acute cardiopulmonary abnormality.'
            }
        ]
    }
}

# Generate report
report = report_generator.generate(
    test_data['abnormality_score'],
    test_data['disease_probs'],
    test_data['retrieved_cases'],
    mode="proposed"
)

print("GENERATED REPORT:")
print("="*60)
print(report)
print("="*60)

## Multi-Model Comparison

In [ ]:
def compare_models(test_case, models_config):
    """
    Compare multiple LLMs on the same case.
    
    Args:
        test_case: Dict with abnormality_score, disease_probs, retrieved_cases
        models_config: Dict of {model_name: llm_client}
    
    Returns:
        dict: {model_name: report}
    """
    results = {}
    
    for model_name, client in models_config.items():
        print(f"\nGenerating with {model_name}...")
        generator = ReportGenerator(client)
        
        report = generator.generate(
            test_case['abnormality_score'],
            test_case['disease_probs'],
            test_case['retrieved_cases'],
            mode="proposed"
        )
        
        results[model_name] = report
    
    return results

In [ ]:
# Example: Compare 3 models
models = {
    'llama3': OllamaClient('llama3'),
    'meditron': OllamaClient('meditron'),
    # 'gpt4': OpenAIClient(api_key='your-key', model='gpt-4')
}

# comparison_results = compare_models(test_data, models)

# for model_name, report in comparison_results.items():
#     print(f"\n{'='*60}")
#     print(f"MODEL: {model_name.upper()}")
#     print('='*60)
#     print(report)

## Save Reports

In [ ]:
def save_report(report, output_path):
    """Save report to file."""
    with open(output_path, 'w') as f:
        f.write(report)
    print(f"✅ Report saved to {output_path}")